# Summary

This notebook ingests Cosmos DB export files from a Unity Catalog volume into a Bronze Delta table using Auto Loader.

- Sets runtime parameters with widgets for source/database names, catalog, batch sizing, corrupt-record handling, and run ID
- Builds landing, checkpoint, and target Bronze table paths/names from those parameters
- Creates a Bronze table to store parsed Cosmos documents as `VARIANT` plus ingestion lineage and JSON-quality metadata
- Reads newline-delimited JSON text files from the landing location with Auto Loader in streaming mode
- Parses each line with `try_parse_json`, extracts common Cosmos fields, and captures corrupt records without failing unless configured
- Writes the stream to the Bronze table with `availableNow=True`, then reports rows ingested and corrupt-row counts for the run
- Includes a final verification query to inspect ingested records


In [0]:
import json, re
from pyspark.sql import functions as F

defaults = {
    "cosmos_database": "pocdb", "cosmos_container": "customers",
    "uc_catalog": "cosmos_poc",
    "max_files_per_batch": "1000",
    "fail_on_corrupt": "false",        # true = fail the task if any invalid JSON lines arrive
    "run_id": "manual",
}
for k, v in defaults.items():
    dbutils.widgets.text(k, v)
p = {k: dbutils.widgets.get(k).strip() for k in defaults}

safe = lambda s: re.sub(r"[^A-Za-z0-9_-]", "_", s)
db, cont     = safe(p["cosmos_database"]), safe(p["cosmos_container"])
SOURCE_KEY   = f"{db}/{cont}"
LANDING_ROOT = f"/Volumes/{p['uc_catalog']}/raw/cosmos_landing/{SOURCE_KEY}"
CHECKPOINT   = f"/Volumes/{p['uc_catalog']}/ops/checkpoints/bronze/{SOURCE_KEY}"
BRONZE_TABLE = f"{p['uc_catalog']}.bronze.{db}__{cont}".replace("-", "_")
print(f"Landing: {LANDING_ROOT}\nCheckpoint: {CHECKPOINT}\nTable: {BRONZE_TABLE}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  id STRING, cosmos_ts BIGINT, cosmos_etag STRING,
  doc VARIANT,
  _corrupt_record STRING, is_valid_json BOOLEAN,
  _source_key STRING, _landing_batch_id BIGINT,
  _source_file STRING, _source_file_mod_time TIMESTAMP,
  _ingested_at TIMESTAMP, _run_id STRING)
COMMENT 'Bronze: Cosmos documents as VARIANT + lineage metadata. Append-only; duplicates resolved in silver.'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

In [0]:
src = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")                  # one raw JSON document per line
        .option("pathGlobFilter", "*.txt")                    # ignore _SUCCESS/_committed/_started markers
        .option("cloudFiles.partitionColumns", "")            # don't turn batch_id=... into a column automatically
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.maxFilesPerTrigger", p["max_files_per_batch"])
        .load(LANDING_ROOT))

bronze = (src
    .filter(F.length(F.trim("value")) > 0)
    .withColumn("doc", F.expr("try_parse_json(value)"))
    .select(
        F.expr("doc:id::string").alias("id"),
        F.expr("doc:_ts::bigint").alias("cosmos_ts"),
        F.expr("doc:_etag::string").alias("cosmos_etag"),
        "doc",
        F.when(F.col("doc").isNull(), F.col("value")).alias("_corrupt_record"),
        F.col("doc").isNotNull().alias("is_valid_json"),
        F.lit(SOURCE_KEY).alias("_source_key"),
        F.regexp_extract("_metadata.file_path", r"batch_id=(\d+)", 1).cast("bigint").alias("_landing_batch_id"),
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_source_file_mod_time"),
        F.current_timestamp().alias("_ingested_at"),
        F.lit(p["run_id"]).alias("_run_id"),
    ))

In [0]:
q = (bronze.writeStream
        .queryName(f"bronze_{cont}")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE))
q.awaitTermination()

rows_in = sum(pr["numInputRows"] for pr in q.recentProgress)
corrupt = spark.sql(f"SELECT count(*) c FROM {BRONZE_TABLE} WHERE NOT is_valid_json AND _run_id = '{p['run_id']}'").first()["c"]
result = {"source": SOURCE_KEY, "run_id": p["run_id"], "rows_ingested": rows_in, "corrupt_rows_for_run_id": corrupt}
print(json.dumps(result))

if corrupt > 0 and p["fail_on_corrupt"].lower() == "true":
    raise RuntimeError(f"{corrupt} invalid JSON line(s) landed in {BRONZE_TABLE}._corrupt_record")

In [0]:
display(spark.sql(f"""
  SELECT id, cosmos_ts, doc:name::string AS name, doc:tier::string AS tier,
         is_valid_json, _landing_batch_id, _ingested_at
  FROM {BRONZE_TABLE} ORDER BY id"""))

In [0]:
T = BRONZE_TABLE

print("1) History of id 2 - bronze is append-only, so both versions exist")
display(spark.sql(f"""
  SELECT id, cosmos_ts, doc:city::string AS city, doc:tier::string AS tier, _landing_batch_id
  FROM {T} WHERE id = '2' ORDER BY cosmos_ts"""))

print("2) Latest version per id - exactly what silver will materialise")
display(spark.sql(f"""
  SELECT id, doc:name::string AS name, doc:city::string AS city, doc:tier::string AS tier, cosmos_ts
  FROM {T}
  QUALIFY row_number() OVER (PARTITION BY id ORDER BY cosmos_ts DESC, _ingested_at DESC) = 1
  ORDER BY cast(id AS INT)"""))

print("3) Odd shapes - nested, arrays, unicode, nulls, escaped newline")
display(spark.sql(f"""
  SELECT id,
         doc:name::string                    AS name,
         doc:address.geo.lat::double         AS lat,
         doc:tags[0]::string                 AS first_tag,
         doc:orders[0].amount::decimal(10,2) AS first_order_amount,
         doc:notes::string                   AS notes,
         doc:active::boolean                 AS active,
         schema_of_variant(doc)              AS inferred_schema
  FROM {T} WHERE id IN ('9', '10')"""))

print("4) Raw layer line count (expect 11):",
      spark.read.text(f"{LANDING_ROOT}/*/*.txt").count())